In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU: No GPU")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [144]:
!nvidia-smi

Thu Oct 22 07:24:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [145]:
import os

print(os.listdir("/kaggle/input"))

['datasets']


In [146]:
import os

print(os.listdir("/kaggle/input/datasets"))

['xxxwwj']


In [147]:
import os

print(os.listdir("/kaggle/input/datasets/xxxwwj"))

['cctsdb2021']


In [148]:
import os

base = "/kaggle/input/datasets/xxxwwj/cctsdb2021"

for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    if level <= 2:
        print("  " * level + os.path.basename(root) + "/")

cctsdb2021/
  CCTSDB2021/
    XML/
    test/
    train/
    测试集天气光照情况分类/
    测试集尺寸大小分类/
    负样本图像/


In [149]:
base = "/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021"

import os

print("TRAIN:")
print(os.listdir(base + "/train")[:20])

print("\nTEST:")
print(os.listdir(base + "/test")[:20])

TRAIN:
['images', 'labels']

TEST:
['images', 'labels']


In [150]:
base = "/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021"

import os

train_images = os.listdir(base + "/train/images")
train_labels = os.listdir(base + "/train/labels")
test_images = os.listdir(base + "/test/images")
test_labels = os.listdir(base + "/test/labels")

print("Train images :", len(train_images))
print("Train labels :", len(train_labels))
print("Test images  :", len(test_images))
print("Test labels  :", len(test_labels))

Train images : 16356
Train labels : 16356
Test images  : 1500
Test labels  : 1500


In [151]:
import os
import random
import shutil

# Original dataset
base = "/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021"

# New subset location
subset = "/kaggle/working/CCTSDB2021_subset"

# Create folders
for folder in [
    "train/images",
    "train/labels",
    "val/images",
    "val/labels"
]:
    os.makedirs(os.path.join(subset, folder), exist_ok=True)

# Get image files
images = [
    f for f in os.listdir(base + "/train/images")
    if f.lower().endswith((".jpg", ".jpeg", ".png"))
]

# Reproducible random selection
random.seed(42)
random.shuffle(images)

# Select 4,500 images
selected = images[:4500]

train_files = selected[:4000]
val_files = selected[4000:4500]

# Copy training files
for img in train_files:
    label = os.path.splitext(img)[0] + ".txt"

    shutil.copy(
        os.path.join(base, "train/images", img),
        os.path.join(subset, "train/images", img)
    )

    shutil.copy(
        os.path.join(base, "train/labels", label),
        os.path.join(subset, "train/labels", label)
    )

# Copy validation files
for img in val_files:
    label = os.path.splitext(img)[0] + ".txt"

    shutil.copy(
        os.path.join(base, "train/images", img),
        os.path.join(subset, "val/images", img)
    )

    shutil.copy(
        os.path.join(base, "train/labels", label),
        os.path.join(subset, "val/labels", label)
    )

print("Subset created successfully!")
print("Training images:", len(os.listdir(subset + "/train/images")))
print("Training labels:", len(os.listdir(subset + "/train/labels")))
print("Validation images:", len(os.listdir(subset + "/val/images")))
print("Validation labels:", len(os.listdir(subset + "/val/labels")))

Subset created successfully!
Training images: 4000
Training labels: 4000
Validation images: 500
Validation labels: 500


In [153]:
%cd /kaggle/working/yolov5

yaml_content = """
train: /kaggle/working/CCTSDB2021_subset/train/images
val: /kaggle/working/CCTSDB2021_subset/val/images

nc: 3
names:
  0: mandatory
  1: prohibitory
  2: warning
"""

with open("data/cctsdb_subset.yaml", "w") as f:
    f.write(yaml_content)

print("Dataset YAML created successfully!")
print(open("data/cctsdb_subset.yaml").read())

/kaggle/working/yolov5
Dataset YAML created successfully!

train: /kaggle/working/CCTSDB2021_subset/train/images
val: /kaggle/working/CCTSDB2021_subset/val/images

nc: 3
names:
  0: mandatory
  1: prohibitory
  2: warning



In [24]:
%cd /kaggle/working/yolov5

with open("test.py", "r") as f:
    code = f.read()

# Disable W&B image collection
code = code.replace(
    "if plots and len(wandb_images) < log_imgs:",
    "if False and plots and len(wandb_images) < log_imgs:"
)

# Disable final W&B logging
code = code.replace(
    "if wandb and wandb.run:",
    "if False:"
)

with open("test.py", "w") as f:
    f.write(code)

print("W&B completely disabled in test.py")

/kaggle/working/yolov5
W&B completely disabled in test.py


In [21]:
%cd /kaggle/working/yolov5

!grep -n "wandb.Image" test.py
!grep -n "import wandb" test.py

/kaggle/working/yolov5
243:            wandb.log({"Validation": [wandb.Image(str(f), caption=f.name) for f in sorted(save_dir.glob('test*.jpg'))]})


In [27]:
%cd /kaggle/working/yolov5

!sed -i "s/torch.load(f, map_location=torch.device('cpu'))/torch.load(f, map_location=torch.device('cpu'), weights_only=False)/" utils/general.py

!grep -n "torch.load(f, map_location=torch.device('cpu')" utils/general.py

/kaggle/working/yolov5
363:    x = torch.load(f, map_location=torch.device('cpu'), weights_only=False)


In [37]:
%cd /kaggle/working/yolov5

!WANDB_DISABLED=true WANDB_MODE=disabled python train.py \
 --img-size 416 \
 --batch-size 32 \
 --epochs 100 \
 --data /kaggle/working/yolov5/data/cctsdb_subset.yaml \
 --cfg /kaggle/working/yolov5/models/yolov5s_proposed.yaml \
 --weights '' \
 --hyp /kaggle/working/yolov5/data/hyp.cctsdb.yaml \
 --name Proposed_Kaggle_Test3 \
 --workers 2 \
 --device 0

/kaggle/working/yolov5
Using torch 2.10.0+cu128 CUDA:0 (Tesla T4, 14911.6875MB)


Namespace(weights='', cfg='/kaggle/working/yolov5/models/yolov5s_proposed.yaml', data='/kaggle/working/yolov5/data/cctsdb_subset.yaml', hyp='/kaggle/working/yolov5/data/hyp.cctsdb.yaml', epochs=100, batch_size=32, img_size=[416, 416], rect=False, resume=False, nosave=False, notest=False, noautoanchor=False, evolve=False, bucket='', cache_images=False, image_weights=False, device='0', multi_scale=False, single_cls=False, adam=False, sync_bn=False, local_rank=-1, log_imgs=16, log_artifacts=False, workers=2, project='runs/train', name='Proposed_Kaggle_Test3', exist_ok=False, quad=False, world_size=1, global_rank=-1, save_dir='runs/train/Proposed_Kaggle_Test35', total_batch_size=32)
Start Tensorboard with "tensorboard --logdir runs/train", view at http://localhost:6006/
Hyperparameters {'lr0': 0.001, 'lrf': 0.2, 'momentum': 0.98, 'weight_decay': 0.001, 'warmup_epochs': 3.0, 'warmup_momentum': 0.8, 'warmup_bia

**COMPLETE CODE**

In [10]:
import os

root = "/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021"

for folder in [
    "train/images",
    "train/labels",
    "test/images",
    "test/labels",
]:
    path = os.path.join(root, folder)
    print(
        folder,
        "FOUND" if os.path.isdir(path) else "MISSING",
        "Items:", len(os.listdir(path)) if os.path.isdir(path) else 0
    )

train/images FOUND Items: 16356
train/labels FOUND Items: 16356
test/images FOUND Items: 1500
test/labels FOUND Items: 1500


In [20]:
import os

repo = "/kaggle/working/yolov5"

print("Repository exists:", os.path.exists(repo))
print("Repository contents:")
print(os.listdir(repo))

Repository exists: True
Repository contents:
['data']


In [21]:
import os

repo = "/kaggle/working/yolov5"

for folder in ["models", "data"]:
    path = os.path.join(repo, folder)
    print(f"\n{folder} exists:", os.path.isdir(path))

    if os.path.isdir(path):
        print(os.listdir(path)[:20])


models exists: False

data exists: True
['cctsdb_val.txt', 'cctsdb_subset.yaml', 'cctsdb_train.txt']


In [23]:
%cd /kaggle/working
!git clone https://github.com/ultralytics/yolov5.git yolov5_repo

/kaggle/working
Cloning into 'yolov5_repo'...
remote: Enumerating objects: 18806, done.
remote: Counting objects: 100% (360/360), done.
remote: Compressing objects: 100% (209/209), done.
remote: Total 18806 (delta 299), reused 151 (delta 151), pack-reused 18446 (from 2)
Receiving objects: 100% (18806/18806), 17.72 MiB | 28.99 MiB/s, done.
Resolving deltas: 100% (12816/12816), done.


In [25]:
import os
import shutil

old_data = "/kaggle/working/yolov5/data"
new_data = "/kaggle/working/yolov5_repo/data"

for name in [
    "cctsdb_train.txt",
    "cctsdb_val.txt",
    "cctsdb_subset.yaml"
]:
    src = os.path.join(old_data, name)
    dst = os.path.join(new_data, name)

    if os.path.exists(src):
        shutil.copy2(src, dst)
        print("Copied:", name)
    else:
        print("Missing:", name)

Copied: cctsdb_train.txt
Copied: cctsdb_val.txt
Copied: cctsdb_subset.yaml


In [27]:
config = "/kaggle/working/yolov5_repo/data/cctsdb_subset.yaml"

with open(config, "w") as f:
    f.write("""train: /kaggle/working/yolov5/data/cctsdb_train.txt
val: /kaggle/working/yolov5/data/cctsdb_val.txt

nc: 3
names: ['prohibitory', 'mandatory', 'warning']
""")

print("Dataset configuration updated!")

Dataset configuration updated!


In [28]:
import os

for name in ["cctsdb_train.txt", "cctsdb_val.txt"]:
    path = f"/kaggle/working/yolov5/data/{name}"
    print(name, "exists:", os.path.isfile(path))

    if os.path.isfile(path):
        with open(path) as f:
            print("Number of image entries:", sum(1 for _ in f))

cctsdb_train.txt exists: True
Number of image entries: 14720
cctsdb_val.txt exists: True
Number of image entries: 1636


In [29]:
import os

for root, dirs, files in os.walk("/kaggle/working"):
    dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

    for name in files:
        if "stormsight" in name.lower() or "bifpn" in name.lower():
            print(os.path.join(root, name))

/kaggle/working/yolov5_repo/models/hub/yolov5-bifpn.yaml


In [31]:
%cd /kaggle/working/yolov5_repo

!grep -R -n "BiFPN_Concat\|CoordinateAttention\|AlphaIoU" \
    models utils --include="*.py"

/kaggle/working/yolov5_repo


In [33]:
repo = "/kaggle/working/yolov5_repo"

with open(f"{repo}/models/yolo.py", "r") as f:
    content = f.read()

print("Model parser found:", "def parse_model" in content)

with open(f"{repo}/train.py", "r") as f:
    content = f.read()

print("Training script exists:", len(content) > 0)

Model parser found: True
Training script exists: True


In [36]:
base = "/kaggle/working/yolov5/data"

def read_paths(filename):
    with open(os.path.join(base, filename)) as f:
        return {line.strip() for line in f if line.strip()}

train_images = read_paths("cctsdb_train.txt")
val_images = read_paths("cctsdb_val.txt")

print("Training images:", len(train_images))
print("Validation images:", len(val_images))
print("Overlapping paths:", len(train_images & val_images))

Training images: 14720
Validation images: 1636
Overlapping paths: 0


In [37]:
import os

dataset = "/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021"

for root, dirs, files in os.walk(dataset):
    level = root.replace(dataset, "").count(os.sep)

    if level <= 3:
        print("  " * level + os.path.basename(root) + "/")
        for name in files[:5]:
            print("  " * (level + 1) + name)

CCTSDB2021/
  voc_label.py
  XML/
    00000.xml
    00001.xml
    00002.xml
    00003.xml
    00004.xml
  test/
    images/
      18992.jpg
      18993.jpg
      18994.jpg
      18995.jpg
      18996.jpg
    labels/
      18992.txt
      18993.txt
      18994.txt
      18995.txt
      18996.txt
  train/
    images/
      00000.jpg
      00001.jpg
      00002.jpg
      00003.jpg
      00004.jpg
    labels/
      00000.txt
      00001.txt
      00002.txt
      00003.txt
      00004.txt
  测试集天气光照情况分类/
    cloud/
      19792.xml
      19793.xml
      19794.xml
      19795.xml
      19796.xml
    foggy/
      18992.xml
      18993.xml
      18994.xml
      18995.xml
      18996.xml
    night/
      19292.xml
      19293.xml
      19294.xml
      19295.xml
      19296.xml
    rain/
      19132.xml
      19133.xml
      19134.xml
      19135.xml
      19136.xml
    snow/
      19032.xml
      19033.xml
      19034.xml
      19035.xml
      19036.xml
    sunny/
      20092.xml
      20093.xml


In [40]:
from collections import Counter
import os

data_dir = "/kaggle/working/yolov5/data"

class_names = {
    0: "prohibitory",
    1: "mandatory",
    2: "warning"
}

for split in ["cctsdb_train.txt", "cctsdb_val.txt"]:
    with open(os.path.join(data_dir, split)) as f:
        images = [line.strip() for line in f if line.strip()]

    counts = Counter()

    for img in images:
        label = os.path.splitext(
            img.replace("/images/", "/labels/")
        )[0] + ".txt"

        with open(label) as f:
            for line in f:
                values = line.split()
                if values:
                    counts[int(values[0])] += 1

    print(f"\n{split}")
    for class_id, name in class_names.items():
        print(f"{name}: {counts[class_id]} objects")


cctsdb_train.txt
prohibitory: 7512 objects
mandatory: 12546 objects

cctsdb_val.txt
prohibitory: 852 objects
mandatory: 1330 objects


In [48]:
%cd /kaggle/working/yolov5_repo
%pip install -q "ultralytics>=8.4.164"

/kaggle/working/yolov5_repo
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 27.4 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 3.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [50]:
import os
import shutil

REPO = "/kaggle/working/yolov5_repo"
os.chdir(REPO)

# Create a backup without overwriting an existing backup
backup = "utils/loss.py.backup"
if not os.path.exists(backup):
    shutil.copy2("utils/loss.py", backup)

# Verify the required imports
from ultralytics.utils.metrics import bbox_iou, smooth_bce

print("Backup created:", os.path.exists(backup))
print("bbox_iou imported:", callable(bbox_iou))
print("smooth_bce imported:", callable(smooth_bce))

Backup created: True
bbox_iou imported: True
smooth_bce imported: True


In [52]:
import os
import shutil

os.chdir("/kaggle/working/yolov5_repo")

loss_path = "utils/loss.py"
backup_path = "utils/loss.py.backup"

# Safety checks
assert os.path.exists(loss_path), "loss.py not found"
assert os.path.exists(backup_path), "Backup not found"

with open(loss_path, "r", encoding="utf-8") as f:
    source = f.read()

helper = '''

def alpha_iou_loss(box1, box2, alpha=3.0, eps=1e-7):
    """Alpha-IoU regression loss using IoU in xywh format."""
    iou = bbox_iou(box1, box2, xywh=True, CIoU=False, eps=eps)
    iou = iou.clamp(min=0.0, max=1.0)
    return 1.0 - iou.pow(alpha)
'''

if "def alpha_iou_loss(" not in source:
    # Insert helper before ComputeLoss
    marker = "class ComputeLoss:"
    assert marker in source, "Could not find ComputeLoss class"
    source = source.replace(marker, helper + "\n\n" + marker, 1)

    # Write atomically so the original file isn't left partially written
    temp_path = loss_path + ".tmp"
    with open(temp_path, "w", encoding="utf-8") as f:
        f.write(source)
    os.replace(temp_path, loss_path)

print("Alpha-IoU helper added:", "def alpha_iou_loss(" in source)

Alpha-IoU helper added: True


In [54]:

import os

os.chdir("/kaggle/working/yolov5_repo")

loss_path = "utils/loss.py"

with open(loss_path, "r", encoding="utf-8") as f:
    source = f.read()

old = """                iou = bbox_iou(pbox, tbox[i], CIoU=True).squeeze()  # iou(prediction, target)
                lbox += (1.0 - iou).mean()  # iou loss
"""

new = """                ciou = bbox_iou(pbox, tbox[i], CIoU=True).squeeze()
                lbox += alpha_iou_loss(pbox, tbox[i], alpha=3.0).mean()

                # Preserve the original CIoU-based objectness target
                iou = ciou
"""

count = source.count(old)
assert count == 1, f"Expected 1 matching block, found {count}"

with open(loss_path, "w", encoding="utf-8") as f:
    f.write(source.replace(old, new, 1))

print("Alpha-IoU integrated successfully!")


Alpha-IoU integrated successfully!


In [55]:
import torch
from utils.loss import ComputeLoss, alpha_iou_loss

print("ComputeLoss imported successfully")
print("Alpha-IoU imported successfully")
print("PyTorch:", torch.__version__)

ComputeLoss imported successfully
Alpha-IoU imported successfully
PyTorch: 2.10.0+cu128


In [59]:

import os

os.chdir("/kaggle/working/yolov5_repo")

files_to_check = [
    "models/yolo.py",
    "models/common.py",
]

terms = [
    "CoordinateAttention",
    "BiFPN_Concat",
    "WeightedFusion",
    "AlphaIoU",
]

for path in files_to_check:
    print(f"\n--- {path} ---")

    with open(path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if any(term in line for term in terms):
            print(f"Line {i+1}: {line.rstrip()}")



--- models/yolo.py ---

--- models/common.py ---


In [148]:

import os
import shutil

os.chdir("/kaggle/working/yolov5_repo")

files = [
    "models/common.py",
    "models/yolo.py",
    "utils/loss.py",
]

for path in files:
    assert os.path.exists(path), f"Missing file: {path}"
    backup = path + ".stormsight_backup"
    if not os.path.exists(backup):
        shutil.copy2(path, backup)
    print(f"{path} -> backup ready")


models/common.py -> backup ready
models/yolo.py -> backup ready
utils/loss.py -> backup ready


In [153]:
import os

repo = "/kaggle/working/yolov5_repo"

print("Repository exists:", os.path.isdir(repo))
print("Current directory:", os.getcwd())

if os.path.isdir(repo):
    os.chdir(repo)
    print("common.py exists:", os.path.isfile("models/common.py"))
    print("yolo.py exists:", os.path.isfile("models/yolo.py"))
    print("loss.py exists:", os.path.isfile("utils/loss.py"))

Repository exists: True
Current directory: /kaggle/working/yolov5_repo
common.py exists: True
yolo.py exists: True
loss.py exists: True


In [155]:
import os

roots = ["/kaggle/working", "/kaggle/input"]

keywords = [
    "common.py",
    "yolo.py",
    "loss.py",
    "stormsight_cctsdb.yaml",
    "backup"
]

for root in roots:
    print(f"\n--- Searching {root} ---")

    for current, dirs, files in os.walk(root):
        # Avoid traversing large hidden folders
        dirs[:] = [d for d in dirs if not d.startswith(".")]

        for filename in files:
            if any(k.lower() in filename.lower() for k in keywords):
                full_path = os.path.join(current, filename)

                if (
                    "common.py" in filename
                    or "yolo.py" in filename
                    or "loss.py" in filename
                    or "stormsight_cctsdb.yaml" in filename
                    or "backup" in filename.lower()
                ):
                    print(full_path)


--- Searching /kaggle/working ---
/kaggle/working/yolov5_repo/models/common.py
/kaggle/working/yolov5_repo/models/yolo.py
/kaggle/working/yolov5_repo/models/common.py.stormsight_backup
/kaggle/working/yolov5_repo/models/yolo.py.stormsight_backup
/kaggle/working/yolov5_repo/utils/loss.py.stormsight_backup
/kaggle/working/yolov5_repo/utils/loss.py
/kaggle/working/yolov5_repo/utils/loss.py.backup
/kaggle/working/yolov5_repo/utils/segment/loss.py
/kaggle/working/yolov5_repo/tests/test_invariant_common.py

--- Searching /kaggle/input ---


In [158]:
import os

repo = "/kaggle/working/yolov5_repo"
os.chdir(repo)

# 1. Check whether custom modules exist anywhere in the repository
for term in ["CoordinateAttention", "BiFPN_Concat"]:
    matches = []

    for root, dirs, files in os.walk(repo):
        dirs[:] = [d for d in dirs if d not in [".git", "__pycache__"]]

        for name in files:
            if name.endswith((".py", ".yaml", ".yml")):
                path = os.path.join(root, name)

                try:
                    with open(path, "r", encoding="utf-8") as f:
                        if term in f.read():
                            matches.append(path)
                except (UnicodeDecodeError, OSError):
                    pass

    print(f"\n{term} found in:")
    print("\n".join(matches) if matches else "Not found")

# 2. Check the existing model configurations
print("\nStormSight model files:")
for name in os.listdir("models"):
    if "storm" in name.lower() or "bifpn" in name.lower():
        print(name)

# 3. Confirm Alpha-IoU is still in the current loss file
with open("utils/loss.py", "r", encoding="utf-8") as f:
    loss = f.read()

print("\nAlpha-IoU reference in current loss.py:",
      "Alpha-IoU" in loss)


CoordinateAttention found in:
Not found

BiFPN_Concat found in:
Not found

StormSight model files:
common.py.stormsight_backup
yolo.py.stormsight_backup

Alpha-IoU reference in current loss.py: True


In [163]:
import torch
from models.stormsight_modules import CoordinateAttention, BiFPN_Concat

x = torch.randn(2, 64, 40, 40)

ca = CoordinateAttention(64)
y = ca(x)

fusion = BiFPN_Concat()
z = fusion([y, y])

print("Input shape:", x.shape)
print("Coordinate Attention output:", y.shape)
print("BiFPN output:", z.shape)
print("Fusion weights:", fusion.w.detach().tolist())

assert y.shape == x.shape
assert z.shape == (2, 128, 40, 40)
print("Module smoke test passed.")

Input shape: torch.Size([2, 64, 40, 40])
Coordinate Attention output: torch.Size([2, 64, 40, 40])
BiFPN output: torch.Size([2, 128, 40, 40])
Fusion weights: [1.0, 1.0]
Module smoke test passed.


In [176]:
import os
import importlib

os.chdir("/kaggle/working/yolov5_repo")

common_path = "models/common.py"
recovery_path = "models/stormsight_modules.py"

# Read the actual class definitions from the recovery file
with open(recovery_path, "r", encoding="utf-8") as f:
    recovery = f.read()

required = ["CoordinateAttention", "BiFPN_Concat"]

for name in required:
    assert f"class {name}" in recovery, f"{name} missing from recovery file"

# Remove any stale Python module cache
import models.common
importlib.invalidate_caches()

# Append definitions only if the actual definitions are missing
with open(common_path, "r", encoding="utf-8") as f:
    common = f.read()

missing = [
    name for name in required
    if f"class {name}" not in common
]

if missing:
    with open(common_path, "a", encoding="utf-8") as f:
        f.write("\n\n" + recovery + "\n")
    print("Added missing classes:", missing)
else:
    print("Class text exists, but the loaded module may be stale.")

print("Please restart the Kaggle session's Python process before importing again.")

Class text exists, but the loaded module may be stale.
Please restart the Kaggle session's Python process before importing again.


In [179]:
import os
import sys
import importlib

os.chdir("/kaggle/working/yolov5_repo")

# Remove cached modules so Python loads the current source
for name in ["models.yolo", "models.common"]:
    sys.modules.pop(name, None)

importlib.invalidate_caches()

# Import the current module from disk
common = importlib.import_module("models.common")

print("Loaded from:", common.__file__)
print("CoordinateAttention:", hasattr(common, "CoordinateAttention"))
print("BiFPN_Concat:", hasattr(common, "BiFPN_Concat"))

assert hasattr(common, "CoordinateAttention")
assert hasattr(common, "BiFPN_Concat")

print("Both custom classes imported successfully!")

Loaded from: /kaggle/working/yolov5_repo/models/common.py
CoordinateAttention: True
BiFPN_Concat: True
Both custom classes imported successfully!


In [180]:
from models.yolo import parse_model

print("YOLOv5 parser import: OK")

YOLOv5 parser import: OK


In [182]:

import os
import shutil
from pathlib import Path

os.chdir("/kaggle/working/yolov5_repo")

yolo_file = Path("models/yolo.py")
source = yolo_file.read_text()

# 1. Register imports
import_marker = "from models.common import ("
assert import_marker in source, "Could not find models.common import block."

if "    CoordinateAttention," not in source:
    source = source.replace(
        import_marker,
        import_marker + "\n    CoordinateAttention,\n    BiFPN_Concat,"
    )

# 2. Register custom modules in parse_model()
parse_marker = '    for i, (f, n, m, args) in enumerate(d["backbone"] + d["head"]):'
assert parse_marker in source, "Could not find parse_model loop."

# Locate the parser's existing module handling
if "elif m is CoordinateAttention:" not in source:
    concat_marker = "        elif m is Concat:"
    if concat_marker not in source:
        concat_marker = "    elif m is Concat:"

    assert concat_marker in source, (
        "Could not locate the Concat parser branch. "
        "Inspect parse_model() before editing."
    )

    indent = concat_marker[:len(concat_marker) - len(concat_marker.lstrip())]
    custom_branches = (
        f"{indent}elif m is CoordinateAttention:\n"
        f"{indent}    c2 = ch[f]\n"
        f"{indent}    args = [c2, *args]\n"
        f"{indent}elif m is BiFPN_Concat:\n"
        f"{indent}    assert isinstance(f, list) and len(f) == 2, "
        '"BiFPN_Concat requires exactly two input layers"\n'
        f"{indent}    c2 = ch[f[0]]\n"
    )
    source = source.replace(concat_marker, custom_branches + concat_marker, 1)

# Save a separate backup before writing
backup = Path("models/yolo.py.before_custom_registration")
if not backup.exists():
    shutil.copy2(yolo_file, backup)

yolo_file.write_text(source)

print("Custom module imports and parser branches registered.")
print("Backup:", backup)


Custom module imports and parser branches registered.
Backup: models/yolo.py.before_custom_registration


In [184]:

import os
import torch
from pathlib import Path

os.chdir("/kaggle/working/yolov5_repo")

from models.common import CoordinateAttention, BiFPN_Concat
from models.yolo import parse_model

source = Path("models/yolo.py").read_text()

print("CoordinateAttention import:", CoordinateAttention.__name__)
print("BiFPN_Concat import:", BiFPN_Concat.__name__)
print("YOLO parser imported successfully:", callable(parse_model))
print(
    "CoordinateAttention parser branch:",
    "elif m is CoordinateAttention:" in source
)
print(
    "BiFPN_Concat parser branch:",
    "elif m is BiFPN_Concat:" in source
)
print("PyTorch version:", torch.__version__)


CoordinateAttention import: CoordinateAttention
BiFPN_Concat import: BiFPN_Concat
YOLO parser imported successfully: True
CoordinateAttention parser branch: True
BiFPN_Concat parser branch: True
PyTorch version: 2.10.0+cu128


In [188]:

from pathlib import Path
import yaml

repo = Path("/kaggle/working/yolov5_repo")
data_cfg = repo / "data/cctsdb_subset.yaml"

with open(data_cfg, "r") as f:
    data = yaml.safe_load(f)

for split in ("train", "val"):
    list_path = Path(data[split])
    print(f"\n{split.upper()} list:", list_path)
    print("List exists:", list_path.exists())

    if list_path.exists():
        paths = [
            line.strip()
            for line in list_path.read_text().splitlines()
            if line.strip()
        ]
        valid = sum(Path(p).is_file() for p in paths)
        print("Images listed:", len(paths))
        print("Valid image paths:", valid)
        print("Missing paths:", len(paths) - valid)

print("\nClasses:", data["names"])
print("Number of classes:", data["nc"])



TRAIN list: /kaggle/working/yolov5/data/cctsdb_train.txt
List exists: True
Images listed: 14720
Valid image paths: 14720
Missing paths: 0

VAL list: /kaggle/working/yolov5/data/cctsdb_val.txt
List exists: True
Images listed: 1636
Valid image paths: 1636
Missing paths: 0

Classes: ['prohibitory', 'mandatory', 'warning']
Number of classes: 3


In [191]:

import os
from pathlib import Path
import torch

os.chdir("/kaggle/working/yolov5_repo")

loss_text = Path("utils/loss.py").read_text()

checks = {
    "Alpha-IoU function defined": "def alpha_iou_loss(" in loss_text,
    "Alpha-IoU called in box loss": "alpha_iou_loss(pbox, tbox[i]" in loss_text,
    "CIoU retained for objectness": "iou = ciou" in loss_text,
}

for name, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}: {name}")

print("\nCUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PASS: Alpha-IoU function defined
PASS: Alpha-IoU called in box loss
PASS: CIoU retained for objectness

CUDA available: True
GPU: Tesla T4


In [192]:

from pathlib import Path
import os

os.chdir("/kaggle/working/yolov5_repo")

path = Path("train.py")
text = path.read_text()

old = "init_seeds(opt.seed + 1 + RANK, deterministic=True)"
new = "init_seeds(opt.seed + 1 + RANK, deterministic=False)"

count = text.count(old)

if count == 1:
    # Keep a backup before editing
    backup = Path("train.py.before_determinism_fix")
    if not backup.exists():
        backup.write_text(text)

    path.write_text(text.replace(old, new, 1))
    print("Fixed: deterministic algorithms disabled for training.")
elif count == 0 and new in text:
    print("Setting is already fixed.")
else:
    print(f"Expected one matching line but found {count}.")
    print("Inspect the init_seeds call before changing anything.")


Fixed: deterministic algorithms disabled for training.


In [297]:

import os
from pathlib import Path
import pandas as pd

os.chdir("/kaggle/working/yolov5_repo")

run_dir = Path("runs/train/stormsight_smoke_test")

# 1. Read recorded metrics
csv_path = run_dir / "results.csv"
if csv_path.exists():
    results = pd.read_csv(csv_path)
    results.columns = results.columns.str.strip()
    print("Training results:")
    print(results.tail(3).to_string(index=False))
else:
    print("results.csv not found")

# 2. Inspect the actual box-loss implementation
loss_text = Path("utils/loss.py").read_text().splitlines()
for i, line in enumerate(loss_text):
    if "alpha_iou_loss(pbox" in line or "iou = ciou" in line:
        start = max(0, i - 3)
        end = min(len(loss_text), i + 4)
        print(f"\n--- loss.py lines {start + 1}-{end} ---")
        print("\n".join(
            f"{j + 1}: {loss_text[j]}"
            for j in range(start, end)
        ))

# 3. Check class distribution in the training labels
from collections import Counter

data_yaml = Path("data/cctsdb_subset.yaml")
import yaml

data = yaml.safe_load(data_yaml.read_text())
class_counts = Counter()

for image_line in Path(data["train"]).read_text().splitlines():
    image_path = Path(image_line.strip())
    label_path = Path(
        str(image_path).replace("/images/", "/labels/")
    ).with_suffix(".txt")

    if label_path.exists():
        for row in label_path.read_text().splitlines():
            values = row.split()
            if len(values) == 5:
                class_counts[int(float(values[0]))] += 1

print("\nTraining label counts:")
for class_id, name in enumerate(data["names"]):
    print(f"{class_id}: {name} = {class_counts[class_id]}")

print("\nCheckpoint exists:",
      (run_dir / "weights/best.pt").exists())


Training results:
 epoch  train/box_loss  train/obj_loss  train/cls_loss  metrics/precision  metrics/recall  metrics/mAP_0.5  metrics/mAP_0.5:0.95  val/box_loss  val/obj_loss  val/cls_loss    x/lr0    x/lr1    x/lr2
     0         0.11266        0.007172        0.027958           0.000015        0.004762         0.000008              0.000002       0.10523      0.007407      0.026463 0.070016 0.003331 0.003331
     0         0.11266        0.007163        0.027957           0.000015        0.004762         0.000008              0.000002       0.10503      0.007888      0.026414 0.070016 0.003331 0.003331

--- loss.py lines 160-166 ---
160:                 pwh = (pwh.sigmoid() * 2) ** 2 * anchors[i]
161:                 pbox = torch.cat((pxy, pwh), 1)  # predicted box
162:                 ciou = bbox_iou(pbox, tbox[i], CIoU=True).squeeze()
163:                 lbox += alpha_iou_loss(pbox, tbox[i], alpha=3.0).mean()
164: 
165:                 # Preserve the original CIoU-based objectness

In [321]:
%cd /kaggle/working/yolov5_repo

import torch
from utils.loss import alpha_iou_loss

# Simulated predicted and target bounding boxes in xywh format
pred = torch.tensor(
    [[0.50, 0.50, 0.10, 0.10],
     [0.30, 0.30, 0.05, 0.05]],
    requires_grad=True
)

target = torch.tensor(
    [[0.52, 0.51, 0.10, 0.10],
     [0.70, 0.70, 0.08, 0.08]]
)

loss = alpha_iou_loss(pred, target, alpha=3.0).mean()
loss.backward()

print("Alpha-IoU loss:", loss.item())
print("Gradients finite:", torch.isfinite(pred.grad).all().item())
print("Gradient magnitude:", pred.grad.abs().mean().item())
print("Gradients:\n", pred.grad)

/kaggle/working/yolov5_repo
Alpha-IoU loss: 0.9110124707221985
Gradients finite: True
Gradient magnitude: 1.3252524137496948
Gradients:
 tensor([[-5.21410, -4.63475, -0.52142, -0.23175],
        [ 0.00000,  0.00000,  0.00000,  0.00000]])


In [322]:
from pathlib import Path
import pandas as pd
import torch

run_dir = Path("runs/train/stormsight_smoke_test")
weights = run_dir / "weights/best.pt"

print("Checkpoint exists:", weights.exists())

if weights.exists():
    checkpoint = torch.load(
        weights, map_location="cpu", weights_only=False
    )
    print("Checkpoint keys:", list(checkpoint.keys()))
    print("Epoch recorded:", checkpoint.get("epoch", "unknown"))

results_path = run_dir / "results.csv"
if results_path.exists():
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    print("\nRecorded training and validation losses:")
    print(df[[
        "train/box_loss", "train/obj_loss", "train/cls_loss",
        "val/box_loss", "val/obj_loss", "val/cls_loss"
    ]].tail().to_string(index=False))

Checkpoint exists: True
Checkpoint keys: ['epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'opt', 'git', 'date']
Epoch recorded: -1

Recorded training and validation losses:
 train/box_loss  train/obj_loss  train/cls_loss  val/box_loss  val/obj_loss  val/cls_loss
        0.11266        0.007172        0.027958       0.10523      0.007407      0.026463
        0.11266        0.007163        0.027957       0.10503      0.007888      0.026414


In [323]:
%cd /kaggle/working/yolov5_repo

import torch
from utils.loss import alpha_iou_loss

target = torch.tensor([[0.50, 0.50, 0.20, 0.20]])

tests = [
    ("Good overlap", [0.52, 0.51, 0.20, 0.20]),
    ("Partial overlap", [0.60, 0.50, 0.20, 0.20]),
    ("No overlap", [0.90, 0.90, 0.20, 0.20]),
]

for label, values in tests:
    pred = torch.tensor(
        [values], dtype=torch.float32, requires_grad=True
    )

    loss = alpha_iou_loss(pred, target, alpha=3.0).mean()
    loss.backward()

    print(f"\n{label}")
    print("Loss:", round(loss.item(), 6))
    print("Gradient magnitude:", round(pred.grad.abs().mean().item(), 6))
    print("Gradients finite:", torch.isfinite(pred.grad).all().item())

/kaggle/working/yolov5_repo

Good overlap
Loss: 0.58363
Gradient magnitude: 6.124497
Gradients finite: True

Partial overlap
Loss: 0.962963
Gradient magnitude: 0.462961
Gradients finite: True

No overlap
Loss: 1.0
Gradient magnitude: 0.0
Gradients finite: True


In [193]:
%cd /kaggle/working/yolov5_repo

from pathlib import Path

val_list = Path("data/cctsdb_val.txt")
sample_list = Path("data/cctsdb_val_sample.txt")

images = [
    line.strip()
    for line in val_list.read_text().splitlines()
    if line.strip()
]

assert images, "No validation images found in the list."

sample_list.write_text("\n".join(images[:20]) + "\n")
print(f"Created sample list with {min(20, len(images))} images.")

/kaggle/working/yolov5_repo
Created sample list with 20 images.


In [325]:
!python detect.py \
  --weights runs/train/stormsight_smoke_test/weights/best.pt \
  --source data/cctsdb_val_sample.txt \
  --imgsz 640 \
  --conf-thres 0.01 \
  --device 0 \
  --max-det 30 \
  --save-txt \
  --save-conf \
  --project runs/predict \
  --name stormsight_check \
  --exist-ok

detect: weights=['runs/train/stormsight_smoke_test/weights/best.pt'], source=data/cctsdb_val_sample.txt, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.01, iou_thres=0.45, max_det=30, device=0, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project=runs/predict, name=stormsight_check, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-561-gaf08ba15 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)

Fusing layers... 
stormsight_cctsdb summary: 157 layers, 8,021,516 parameters, 0 gradients, 20.1 GFLOPs
image 1/20 /kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train/images/00474.jpg: 480x640 (no detections), 57.9ms
image 2/20 /kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train/images/00753.jpg: 640x448 (no detections), 42.2ms
image 3/20 /kaggle/input/datasets/xxxwwj/cc

In [327]:
%cd /kaggle/working/yolov5_repo

!python detect.py \
  --weights runs/train/stormsight_smoke_test/weights/best.pt \
  --source data/cctsdb_val_sample.txt \
  --data data/cctsdb_subset.yaml \
  --imgsz 640 \
  --conf-thres 0.001 \
  --iou-thres 0.45 \
  --device 0 \
  --max-det 30 \
  --save-txt \
  --save-conf \
  --project runs/predict \
  --name stormsight_lowconf \
  --exist-ok

/kaggle/working/yolov5_repo
detect: weights=['runs/train/stormsight_smoke_test/weights/best.pt'], source=data/cctsdb_val_sample.txt, data=data/cctsdb_subset.yaml, imgsz=[640, 640], conf_thres=0.001, iou_thres=0.45, max_det=30, device=0, view_img=False, save_txt=True, save_format=0, save_csv=False, save_conf=True, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, update=False, project=runs/predict, name=stormsight_lowconf, exist_ok=True, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False, vid_stride=1
YOLOv5 🚀 v7.0-561-gaf08ba15 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)

Fusing layers... 
stormsight_cctsdb summary: 157 layers, 8,021,516 parameters, 0 gradients, 20.1 GFLOPs
image 1/20 /kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train/images/00474.jpg: 480x640 30 mandatorys, 56.6ms
image 2/20 /kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train/images/00753.jpg: 640x448 30 mandatorys, 44.5ms
image 3/20

In [194]:
from pathlib import Path

out = Path("runs/predict/stormsight_lowconf")
labels = out / "labels"

files = list(labels.glob("*.txt")) if labels.exists() else []

print("Images checked: 20")
print("Images with prediction files:", len(files))

total = 0
for file in files:
    rows = [x for x in file.read_text().splitlines() if x.strip()]
    total += len(rows)

print("Total detections:", total)

for file in files[:3]:
    print(f"\n{file.name}")
    print(file.read_text()[:1000])

Images checked: 20
Images with prediction files: 0
Total detections: 0


In [329]:
%cd /kaggle/working/yolov5_repo

!WANDB_MODE=disabled python train.py \
  --weights '' \
  --cfg models/stormsight_cctsdb.yaml \
  --data data/cctsdb_subset.yaml \
  --hyp data/hyps/hyp.scratch-low.yaml \
  --epochs 20 \
  --batch-size 8 \
  --imgsz 640 \
  --device 0 \
  --workers 2 \
  --name stormsight_alpha_pilot \
  --exist-ok \
  --save-period 1

/kaggle/working/yolov5_repo
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
train: weights=, cfg=models/stormsight_cctsdb.yaml, data=data/cctsdb_subset.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=20, batch_size=8, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=0, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=2, project=runs/train, name=stormsight_alpha_pilot, exist_ok=True, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1
github: up to date with https://github.com/ultralytics/yolov5 ✅
YOLOv5 🚀 v7.0-561-gaf08ba15 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912M

In [196]:
%cd /kaggle/working/yolov5_repo

import os
import yaml

cfg = {
    "nc": 3,
    "depth_multiple": 0.33,
    "width_multiple": 0.50,
    "anchors": [
        [4, 5, 6, 8, 10, 12],
        [15, 19, 23, 30, 39, 45],
        [58, 68, 86, 115, 132, 162],
        [186, 220, 280, 314, 385, 440],
    ],
    "backbone": [
        [-1, 1, "Conv", [64, 6, 2, 2]],
        [-1, 1, "Conv", [128, 3, 2]],
        [-1, 3, "C3", [128]],
        [-1, 1, "Conv", [256, 3, 2]],
        [-1, 6, "C3", [256]],
        [-1, 1, "CoordinateAttention", []],
        [-1, 1, "Conv", [512, 3, 2]],
        [-1, 9, "C3", [512]],
        [-1, 1, "CoordinateAttention", []],
        [-1, 1, "Conv", [1024, 3, 2]],
        [-1, 3, "C3", [1024]],
        [-1, 1, "SPPF", [1024, 5]],
        [-1, 1, "CoordinateAttention", []],
    ],
    "head": [
        [-1, 1, "Conv", [512, 1, 1]],
        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],
        [[-1, 8], 1, "BiFPN_Concat", [1]],
        [-1, 3, "C3", [512, False]],
        [-1, 1, "Conv", [256, 1, 1]],
        [-1, 1, "nn.Upsample", [None, 2, "nearest"]],
        [[-1, 5], 1, "BiFPN_Concat", [1]],
        [-1, 3, "C3", [256, False]],
        [-1, 1, "Conv", [256, 3, 2]],
        [[-1, 17], 1, "BiFPN_Concat", [1]],
        [-1, 3, "C3", [512, False]],
        [-1, 1, "Conv", [512, 3, 2]],
        [[-1, 13], 1, "BiFPN_Concat", [1]],
        [-1, 3, "C3", [1024, False]],
        [[20, 23, 26], 1, "Detect", ["nc", "anchors"]],
    ],
}

path = "models/stormsight_cctsdb.yaml"

with open(path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False)

print("Created:", os.path.abspath(path))
print("File exists:", os.path.isfile(path))

/kaggle/working/yolov5_repo
Created: /kaggle/working/yolov5_repo/models/stormsight_cctsdb.yaml
File exists: True


In [197]:
import yaml
from pathlib import Path

path = Path("models/stormsight_cctsdb.yaml")
cfg = yaml.safe_load(path.read_text())

print("YAML valid:", isinstance(cfg, dict))
print("Backbone layers:", len(cfg["backbone"]))
print("Head layers:", len(cfg["head"]))
print("Classes:", cfg["nc"])
print("Detection layer:", cfg["head"][-1])

YAML valid: True
Backbone layers: 13
Head layers: 15
Classes: 3
Detection layer: [[20, 23, 26], 1, 'Detect', ['nc', 'anchors']]


In [200]:
%cd /kaggle/working/yolov5_repo

from pathlib import Path
import yaml

cfg_path = Path("models/stormsight_cctsdb.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

# Fix the C3 output channels after BiFPN concatenations.
# The concatenated features have 512 channels at layer 15,
# and 256 channels at layer 19.
cfg["head"][1] = [-1, 1, "nn.Upsample", [None, 2, "nearest"]]

# C3 after first BiFPN concat: input 512 -> output 256
cfg["head"][3] = [-1, 3, "C3", [256, False]]

# C3 after second BiFPN concat: input 256 -> output 128
cfg["head"][7] = [-1, 3, "C3", [128, False]]

# C3 after third BiFPN concat: input 256 -> output 256
cfg["head"][10] = [-1, 3, "C3", [256, False]]

# C3 after fourth BiFPN concat: input 512 -> output 512
cfg["head"][13] = [-1, 3, "C3", [512, False]]

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))

print("Updated YAML:", cfg_path)
print("YAML valid:", isinstance(yaml.safe_load(cfg_path.read_text()), dict))

/kaggle/working/yolov5_repo
Updated YAML: models/stormsight_cctsdb.yaml
YAML valid: True


In [204]:
%cd /kaggle/working/yolov5_repo

from pathlib import Path
import yaml
import importlib
import models.yolo as yolo

# 1. Fix BiFPN channel tracking in the parser
parser_path = Path("models/yolo.py")
source = parser_path.read_text()

old = """elif m is BiFPN_Concat:
            assert isinstance(f, list) and len(f) == 2, "BiFPN_Concat requires exactly two input layers"
            c2 = ch[f[0]]"""

new = """elif m is BiFPN_Concat:
            assert isinstance(f, list) and len(f) == 2, "BiFPN_Concat requires exactly two input layers"
            c2 = sum(ch[x] for x in f)"""

count = source.count(old)
assert count == 1, f"Expected one BiFPN parser block, found {count}"

parser_path.write_text(source.replace(old, new, 1))
print("BiFPN parser channel tracking corrected.")

# 2. Match three detection outputs with three anchor groups
cfg_path = Path("models/stormsight_cctsdb.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

cfg["anchors"] = [
    [15, 19, 23, 30, 39, 45],
    [58, 68, 86, 115, 132, 162],
    [186, 220, 280, 314, 385, 440],
]

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Anchor groups:", len(cfg["anchors"]))

# 3. Reload parser so this notebook uses the updated code
importlib.reload(yolo)
print("Updated YOLO parser loaded.")

/kaggle/working/yolov5_repo
BiFPN parser channel tracking corrected.
Anchor groups: 3
Updated YOLO parser loaded.


In [205]:
import torch
from models.yolo import Model

model = Model("models/stormsight_cctsdb.yaml", ch=3, nc=3)
model.eval()

print("\nModel built successfully.")
print("Detection scales:", model.model[-1].nl)
print("Classes:", model.model[-1].nc)

with torch.no_grad():
    output = model(torch.zeros(1, 3, 640, 640))

print("Forward pass successful.")
print("Output type:", type(output).__name__)


                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1      3352  models.common.CoordinateAttention       [128]                         
  6                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  7                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  8                -1  1      6680  

In [206]:
%cd /kaggle/working/yolov5_repo

!WANDB_MODE=disabled python train.py \
  --weights '' \
  --cfg models/stormsight_cctsdb.yaml \
  --data data/cctsdb_subset.yaml \
  --hyp data/hyps/hyp.scratch-low.yaml \
  --epochs 40 \
  --batch-size 8 \
  --imgsz 640 \
  --device 0 \
  --workers 2 \
  --name stormsight_one_epoch_test \
  --exist-ok \
  --save-period 1

/kaggle/working/yolov5_repo
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
train: weights=, cfg=models/stormsight_cctsdb.yaml, data=data/cctsdb_subset.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=40, batch_size=8, imgsz=640, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=0, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=2, project=runs/train, name=stormsight_one_epoch_test, exist_ok=True, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1
github: up to date with https://github.com/ultralytics/yolov5 ✅
YOLOv5 🚀 v7.0-561-gaf08ba15 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 149

In [26]:
from pathlib import Path

root = Path("/kaggle/working/stormsight_checkpoints")

for p in root.rglob("best.pt"):
    print("Best checkpoint:", p, "Size (MB):", round(p.stat().st_size / 1e6, 2))

for p in root.rglob("last.pt"):
    print("Last checkpoint:", p, "Size (MB):", round(p.stat().st_size / 1e6, 2))

In [27]:

from pathlib import Path

search_roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

found = []

for root in search_roots:
    if not root.exists():
        continue

    print(f"\nSearching: {root}")

    for p in root.rglob("*.pt"):
        try:
            size_mb = p.stat().st_size / (1024 * 1024)
            found.append(p)
            print(f"  {p}  ({size_mb:.2f} MB)")
        except OSError:
            pass

if not found:
    print("\nNo .pt checkpoint files found.")
    print("We may need to restore a saved checkpoint or retrain.")
else:
    print(f"\nFound {len(found)} checkpoint file(s).")



Searching: /kaggle/working

Searching: /kaggle/input

No .pt checkpoint files found.
We may need to restore a saved checkpoint or retrain.


In [28]:
%cd /kaggle/working/yolov5_repo

!python train.py \
  --cfg models/stormsight_cctsdb.yaml \
  --data /kaggle/working/stormsight_data/stormsight_data.yaml \
  --weights "" \
  --hyp data/hyps/hyp.scratch-low.yaml \
  --epochs 1 \
  --batch-size 4 \
  --img 320 \
  --workers 2 \
  --device 0 \
  --project /kaggle/working/stormsight_checkpoints \
  --name stormsight_smoke \
  --save-period 1 \
  --exist-ok

[Errno 2] No such file or directory: '/kaggle/working/yolov5_repo'
/kaggle/working
python3: can't open file '/kaggle/working/train.py': [Errno 2] No such file or directory


In [29]:

from pathlib import Path

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n===== {root} =====")

    if not root.exists():
        print("Directory not found")
        continue

    for item in list(root.iterdir())[:30]:
        print(("DIR " if item.is_dir() else "FILE"), item)

print("\n===== Searching for YOLOv5 train.py =====")

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    if root.exists():
        for p in root.rglob("train.py"):
            if p.name == "train.py" and (
                p.parent.name == "yolov5_repo"
                or (p.parent / "models").exists()
            ):
                print("Possible repository:", p.parent)



===== /kaggle/working =====
DIR  /kaggle/working/.virtual_documents

===== /kaggle/input =====
DIR  /kaggle/input/datasets

===== Searching for YOLOv5 train.py =====


In [30]:

from pathlib import Path
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nKaggle input datasets:")
for p in Path("/kaggle/input/datasets").iterdir():
    print(p)

print("\nSearching for CCTSDB2021:")
for p in Path("/kaggle/input").rglob("CCTSDB2021"):
    print(p)


GPU available: True
GPU: Tesla T4

Kaggle input datasets:
/kaggle/input/datasets/xxxwwj

Searching for CCTSDB2021:
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021


In [31]:
%cd /kaggle/working
!git clone https://github.com/ultralytics/yolov5.git yolov5_repo

/kaggle/working
Cloning into 'yolov5_repo'...
remote: Enumerating objects: 18806, done.
remote: Counting objects: 100% (363/363), done.
remote: Compressing objects: 100% (212/212), done.
remote: Total 18806 (delta 303), reused 151 (delta 151), pack-reused 18443 (from 2)
Receiving objects: 100% (18806/18806), 17.72 MiB | 21.20 MiB/s, done.
Resolving deltas: 100% (12817/12817), done.


In [32]:
from pathlib import Path

repo = Path("/kaggle/working/yolov5_repo")
print("Repository exists:", repo.exists())
print("Training script exists:", (repo / "train.py").exists())
print("Model directory exists:", (repo / "models").exists())

Repository exists: True
Training script exists: True
Model directory exists: True


In [33]:

from pathlib import Path
import subprocess

repo = Path("/kaggle/working/yolov5_repo")
dataset = Path("/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021")

print("=== YOLOv5 REPOSITORY ===")
print("Repository exists:", repo.exists())
print("train.py exists:", (repo / "train.py").exists())

if repo.exists():
    result = subprocess.run(
        ["git", "-C", str(repo), "log", "-1", "--oneline"],
        capture_output=True, text=True
    )
    print("Version:", result.stdout.strip())

print("\n=== DATASET ROOT ===")
print("Dataset exists:", dataset.exists())

if dataset.exists():
    print("\nTop-level contents:")
    for p in sorted(dataset.iterdir()):
        print(("DIR  " if p.is_dir() else "FILE "), p.name)

    print("\nImage/annotation directory candidates:")
    for p in dataset.rglob("*"):
        if p.is_dir() and any(
            word in p.name.lower()
            for word in ["image", "label", "annot", "train", "test", "val"]
        ):
            print(p)

    print("\nSample files (first 30):")
    count = 0
    for p in dataset.rglob("*"):
        if p.is_file():
            print(p)
            count += 1
            if count >= 30:
                break


=== YOLOv5 REPOSITORY ===
Repository exists: True
train.py exists: True
Version: af08ba15 Pin CI Python and use AI failure summaries (#13885)

=== DATASET ROOT ===
Dataset exists: True

Top-level contents:
DIR   XML
DIR   test
DIR   train
FILE  voc_label.py
DIR   测试集天气光照情况分类
DIR   测试集尺寸大小分类
DIR   负样本图像

Image/annotation directory candidates:
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/test
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/test/labels
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/test/images
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train/labels
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/train/images

Sample files (first 30):
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/voc_label.py
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/负样本图像/01600.jpg
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/负样本图像/01850.jpg
/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021/负样本图

In [34]:

from pathlib import Path
from collections import Counter
import random

root = Path("/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021")

CLASS_NAMES = ["mandatory", "prohibitory", "warning"]
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def inspect_split(split):
    image_dir = root / split / "images"
    label_dir = root / split / "labels"

    if not image_dir.exists() or not label_dir.exists():
        print(f"\n{split}: images/labels directory missing")
        return

    images = sorted(
        p for p in image_dir.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )
    labels = list(label_dir.rglob("*.txt"))
    label_map = {p.stem: p for p in labels}
    image_stems = {p.stem for p in images}

    missing = [p.name for p in images if p.stem not in label_map]
    extra = [p.name for p in labels if p.stem not in image_stems]

    class_counts = Counter()
    invalid_lines = []
    empty_labels = 0
    object_count = 0

    for image in images:
        label = label_map.get(image.stem)
        if label is None:
            continue

        lines = label.read_text(errors="ignore").splitlines()
        if not lines:
            empty_labels += 1

        for line_no, line in enumerate(lines, 1):
            parts = line.split()
            if len(parts) != 5:
                invalid_lines.append((label.name, line_no, line))
                continue
            try:
                cls = int(parts[0])
                x, y, w, h = map(float, parts[1:])
                if cls not in range(3) or not all(
                    0 <= v <= 1 for v in (x, y, w, h)
                ) or w <= 0 or h <= 0:
                    invalid_lines.append((label.name, line_no, line))
                    continue
                class_counts[cls] += 1
                object_count += 1
            except ValueError:
                invalid_lines.append((label.name, line_no, line))

    print(f"\n=== {split.upper()} ===")
    print("Images:", len(images))
    print("Label files:", len(labels))
    print("Images without matching labels:", len(missing))
    print("Labels without matching images:", len(extra))
    print("Empty label files:", empty_labels)
    print("Valid annotated objects:", object_count)
    print("Class counts:", {
        CLASS_NAMES[c]: class_counts[c] for c in range(3)
    })
    print("Invalid annotation lines:", len(invalid_lines))
    if missing[:5]:
        print("Missing-label examples:", missing[:5])
    if invalid_lines[:5]:
        print("Invalid-line examples:", invalid_lines[:5])

    # Show one real annotation for format confirmation
    for image in images:
        label = label_map.get(image.stem)
        if label and label.stat().st_size > 0:
            print("Sample image:", image.name)
            print("Sample label:", label.name)
            print("First label lines:", label.read_text().splitlines()[:3])
            break

inspect_split("train")
inspect_split("test")

print("\nExpected mapping:")
for i, name in enumerate(CLASS_NAMES):
    print(f"{i}: {name}")



=== TRAIN ===
Images: 16356
Label files: 16356
Images without matching labels: 0
Labels without matching images: 0
Empty label files: 0
Valid annotated objects: 26838
Class counts: {'mandatory': 8364, 'prohibitory': 13876, 'warning': 4598}
Invalid annotation lines: 0
Sample image: 00000.jpg
Sample label: 00000.txt
First label lines: ['2 0.5283203125 0.5084635416666666 0.02734375 0.03515625']

=== TEST ===
Images: 1500
Label files: 1500
Images without matching labels: 0
Labels without matching images: 0
Empty label files: 0
Valid annotated objects: 3228
Class counts: {'mandatory': 718, 'prohibitory': 2177, 'warning': 333}
Invalid annotation lines: 0
Sample image: 18992.jpg
Sample label: 18992.txt
First label lines: ['1 0.5943396226415094 0.44375 0.011792452830188678 0.020833333333333332', '1 0.6869103773584906 0.44479166666666664 0.01061320754716981 0.01875']

Expected mapping:
0: mandatory
1: prohibitory
2: warning


In [35]:

from pathlib import Path
import random
import yaml

ROOT = Path(
    "/kaggle/input/datasets/xxxwwj/cctsdb2021/CCTSDB2021"
)
OUT = Path("/kaggle/working/stormsight_data")
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def get_images(folder):
    return sorted(
        str(p.resolve())
        for p in folder.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    )

train_images = get_images(ROOT / "train" / "images")
test_images = get_images(ROOT / "test" / "images")

assert train_images, "No training images found."
assert test_images, "No test images found."

# Ensure every image has a corresponding YOLO annotation.
for split, images in [("train", train_images), ("test", test_images)]:
    labels_dir = ROOT / split / "labels"
    missing = [
        p for p in images
        if not (labels_dir / f"{Path(p).stem}.txt").exists()
    ]
    assert not missing, f"{split}: {len(missing)} missing labels"

# Deterministic 90/10 split; the test set remains untouched.
rng = random.Random(SEED)
rng.shuffle(train_images)

n_val = round(len(train_images) * 0.10)
val_images = train_images[:n_val]
train_images = train_images[n_val:]

def write_list(filename, paths):
    (OUT / filename).write_text(
        "\n".join(paths) + "\n", encoding="utf-8"
    )

write_list("train.txt", train_images)
write_list("val.txt", val_images)
write_list("test.txt", test_images)

data_yaml = {
    "path": str(OUT),
    "train": str(OUT / "train.txt"),
    "val": str(OUT / "val.txt"),
    "test": str(OUT / "test.txt"),
    "nc": 3,
    "names": ["mandatory", "prohibitory", "warning"],
}

yaml_path = OUT / "stormsight_data.yaml"
yaml_path.write_text(
    yaml.safe_dump(data_yaml, sort_keys=False),
    encoding="utf-8"
)

print("Dataset configuration created:", yaml_path)
print("Training images:", len(train_images))
print("Validation images:", len(val_images))
print("Test images:", len(test_images))
print("\nYAML:\n", yaml_path.read_text())
print("\nTrain/validation overlap:", len(set(train_images) & set(val_images)))
print("Train/test overlap:", len(set(train_images) & set(test_images)))
print("All paths exist:", all(Path(p).exists()
      for p in train_images + val_images + test_images))


Dataset configuration created: /kaggle/working/stormsight_data/stormsight_data.yaml
Training images: 14720
Validation images: 1636
Test images: 1500

YAML:
 path: /kaggle/working/stormsight_data
train: /kaggle/working/stormsight_data/train.txt
val: /kaggle/working/stormsight_data/val.txt
test: /kaggle/working/stormsight_data/test.txt
nc: 3
names:
- mandatory
- prohibitory
- warning


Train/validation overlap: 0
Train/test overlap: 0
All paths exist: True


In [36]:

from pathlib import Path
import re
import torch

repo = Path("/kaggle/working/yolov5_repo")

assert (repo / "models/common.py").exists()
assert (repo / "models/yolo.py").exists()
assert (repo / "utils/loss.py").exists()
assert (repo / "utils/general.py").exists()

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

def show_matches(path, patterns, context=4):
    lines = path.read_text(errors="replace").splitlines()
    matched = set()

    for i, line in enumerate(lines):
        if any(re.search(p, line) for p in patterns):
            matched.update(
                range(max(0, i-context), min(len(lines), i+context+1))
            )

    print(f"\n===== {path.relative_to(repo)} =====")
    last = -2
    for i in sorted(matched):
        if i > last + 1:
            print("...")
        print(f"{i+1:4}: {lines[i]}")
        last = i

show_matches(
    repo / "models/common.py",
    [r"^class SPPF", r"^class Concat", r"^class DetectMultiBackend"],
    context=3
)

show_matches(
    repo / "models/yolo.py",
    [r"^from models\.common import", r"^def parse_model", r"^class Detect",
     r"^class Model", r"isinstance\(m, "],
    context=5
)

show_matches(
    repo / "utils/loss.py",
    [r"^class ComputeLoss", r"bbox_iou", r"CIoU", r"lbox \+="],
    context=5
)

show_matches(
    repo / "utils/general.py",
    [r"^def init_seeds", r"use_deterministic_algorithms"],
    context=5
)

print("\n===== Existing StormSight files =====")
for p in [
    repo / "models/stormsight_cctsdb.yaml",
    repo / "models/common.py",
]:
    print(p, "exists:", p.exists())

print("\nInspection complete. No files were changed.")


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4

===== models/common.py =====
...
 315:             return self.cv2(torch.cat([x] + [m(x) for m in self.m], 1))
 316: 
 317: 
 318: class SPPF(nn.Module):
 319:     """Implements a fast Spatial Pyramid Pooling (SPPF) layer for efficient feature extraction in YOLOv5 models."""
 320: 
 321:     def __init__(self, c1, c2, k=5):
...
 440:         return x.view(b, c // s**2, h * s, w * s)  # x(1,16,160,160)
 441: 
 442: 
 443: class Concat(nn.Module):
 444:     """Concatenates tensors along a specified dimension for efficient tensor manipulation in neural networks."""
 445: 
 446:     def __init__(self, dimension=1):
...
 453:         return torch.cat(x, self.d)
 454: 
 455: 
 456: class DetectMultiBackend(nn.Module):
 457:     """YOLOv5 MultiBackend class for inference on various backends including PyTorch, ONNX, TensorRT, and more."""
 458: 
 459:     def __init__(self, weights="yolov5s.pt", device=torch.device("cpu"), dnn=False, da

In [37]:
from pathlib import Path

repo = Path("/kaggle/working/yolov5_repo")

def print_section(file, start, end):
    lines = (repo / file).read_text().splitlines()
    print(f"\n{'='*20} {file}:{start}-{end} {'='*20}")
    for i in range(start - 1, min(end, len(lines))):
        print(f"{i+1:4}: {lines[i]}")

print_section("models/yolo.py", 28, 65)
print_section("models/yolo.py", 375, 465)
print_section("models/common.py", 1, 45)
print_section("utils/loss.py", 101, 180)
print_section("utils/general.py", 168, 185)

print("\nInspection finished. No files were changed.")


==================== models/yolo.py:28-65 ====================
  28: from models.common import (
  29:     C3,
  30:     C3SPP,
  31:     C3TR,
  32:     SPP,
  33:     SPPF,
  34:     Bottleneck,
  35:     BottleneckCSP,
  36:     C3Ghost,
  37:     C3x,
  38:     Classify,
  39:     Concat,
  40:     Contract,
  41:     Conv,
  42:     CrossConv,
  43:     DetectMultiBackend,
  44:     DWConv,
  45:     DWConvTranspose2d,
  46:     Expand,
  47:     Focus,
  48:     GhostBottleneck,
  49:     GhostConv,
  50:     Proto,
  51: )
  52: from models.experimental import MixConv2d
  53: from utils.autoanchor import check_anchor_order
  54: from utils.general import LOGGER, check_version, check_yaml, colorstr, make_divisible, print_args
  55: from utils.torch_utils import (
  56:     fuse_conv_and_bn,
  57:     initialize_weights,
  58:     model_info,
  59:     profile,
  60:     scale_img,
  61:     select_device,
  62:     time_sync,
  63: )
  64: 
  65: try:

==================== model

In [40]:
from pathlib import Path
import re

repo = Path("/kaggle/working/yolov5_repo")

# Check whether the earlier patch partially completed.
common = (repo / "models/common.py").read_text()
yolo = (repo / "models/yolo.py").read_text()
loss = (repo / "utils/loss.py").read_text()

print("CoordinateAttention added:", "class CoordinateAttention" in common)
print("BiFPN added:", "class BiFPN" in common)
print("CoordinateAttention parser entry:", "elif m is CoordinateAttention:" in yolo)
print("BiFPN parser entry:", "elif m is BiFPN:" in yolo)

print("\n--- Current box-loss lines ---")
for i, line in enumerate(loss.splitlines(), 1):
    if "bbox_iou" in line or "lbox +=" in line:
        print(f"{i}: {line!r}")

print("\n--- Current deterministic setting ---")
for i, line in enumerate((repo / "utils/general.py").read_text().splitlines(), 1):
    if "use_deterministic_algorithms" in line:
        print(f"{i}: {line}")

CoordinateAttention added: True
BiFPN added: True
CoordinateAttention parser entry: True
BiFPN parser entry: True

--- Current box-loss lines ---
6: 'from ultralytics.utils.metrics import bbox_iou, smooth_bce'
153: '                iou = bbox_iou(pbox, tbox[i], CIoU=False).squeeze().clamp(0, 1)  # Alpha-IoU input IoU, alpha=3'
154: '                lbox += (1.0 - iou.pow(3.0)).mean()  # Alpha-IoU, alpha=3'

--- Current deterministic setting ---
180:         torch.use_deterministic_algorithms(True, warn_only=True)


In [42]:
%cd /kaggle/working/yolov5_repo

from pathlib import Path

p = Path("models/common.py")
src = p.read_text()

if "import torch.nn.functional as F" not in src:
    src = src.replace(
        "from torch import nn",
        "from torch import nn\nimport torch.nn.functional as F",
        1
    )
    p.write_text(src)

print("Functional import added:", "import torch.nn.functional as F" in src)

# Reload the updated module.
import importlib
import models.common as common
import models.yolo as yolo
importlib.reload(common)
importlib.reload(yolo)

import torch
from models.common import CoordinateAttention, BiFPN
from models.yolo import DetectionModel
from utils.loss import ComputeLoss

ca = CoordinateAttention(64)
fusion = BiFPN([32, 64], 32)

a = torch.randn(2, 32, 20, 20, requires_grad=True)
x = torch.randn(2, 64, 20, 20, requires_grad=True)

out = fusion([a, x])
assert out.shape == (2, 32, 20, 20), out.shape
out.mean().backward()

assert a.grad is not None and torch.isfinite(a.grad).all()
assert x.grad is not None and torch.isfinite(x.grad).all()

ca_out = ca(torch.randn(2, 64, 20, 20))
assert ca_out.shape == (2, 64, 20, 20)

print("Coordinate Attention: PASS")
print("BiFPN-style fusion forward/backward: PASS")
print("DetectionModel import: PASS")
print("ComputeLoss import: PASS")
print("All component checks passed.")

/kaggle/working/yolov5_repo
Functional import added: True
Coordinate Attention: PASS
BiFPN-style fusion forward/backward: PASS
DetectionModel import: PASS
ComputeLoss import: PASS
All component checks passed.


In [43]:

%cd /kaggle/working/yolov5_repo

from pathlib import Path
import yaml
import torch
import importlib

# Write the integrated StormSight architecture.
model_yaml = {
    "nc": 3,
    "depth_multiple": 0.33,
    "width_multiple": 0.50,
    "anchors": [
        [5, 6, 8, 14, 15, 11],          # P2 / stride 4
        [12, 24, 24, 20, 24, 40],       # P3 / stride 8
        [36, 28, 40, 60, 70, 50],       # P4 / stride 16
        [80, 100, 120, 140, 200, 200],  # P5 / stride 32
    ],
    "backbone": [
        [-1, 1, "Conv", [64, 6, 2, 2]],
        [-1, 1, "Conv", [128, 3, 2]],
        [-1, 3, "C3", [128]],
        [-1, 1, "Conv", [256, 3, 2]],
        [-1, 6, "C3", [256]],
        [-1, 1, "Conv", [512, 3, 2]],
        [-1, 9, "C3", [512]],
        [-1, 1, "Conv", [1024, 3, 2]],
        [-1, 3, "C3", [1024]],
        [-1, 1, "SPPF", [1024, 5]],
        [-1, 1, "CoordinateAttention", [32]],
    ],
    "head": [
        [[2, 4, 6, 10], 1, "BiFPN", [128]],
        [-1, 3, "C3", [128, False]],

        [[4, 6, 10, 12], 1, "BiFPN", [256]],
        [-1, 3, "C3", [256, False]],

        [[6, 10, 14, 12], 1, "BiFPN", [512]],
        [-1, 3, "C3", [512, False]],

        [[10, 16, 14, 12], 1, "BiFPN", [1024]],
        [-1, 3, "C3", [1024, False]],

        [[12, 14, 16, 18], 1, "Detect", ["nc", "anchors"]],
    ],
}

cfg_path = Path("models/stormsight_cctsdb.yaml")
cfg_path.write_text(yaml.safe_dump(model_yaml, sort_keys=False))
print("Created:", cfg_path.resolve())

# Reload modules so the latest custom classes are registered.
import models.common as common
import models.yolo as yolo
importlib.reload(common)
importlib.reload(yolo)

# Build and test the full model.
model = yolo.DetectionModel(str(cfg_path), ch=3, nc=3)
model.train()

dummy = torch.zeros(1, 3, 256, 256)
with torch.no_grad():
    outputs = model(dummy)

print("\nModel built successfully.")
print("Detection strides:", model.stride.tolist())
print("Detection output shapes:")
for i, output in enumerate(outputs):
    print(f"Scale {i + 1}: {tuple(output.shape)}")

expected_strides = [4, 8, 16, 32]
assert model.stride.tolist() == expected_strides, model.stride.tolist()
assert len(outputs) == 4, f"Expected 4 detection scales, got {len(outputs)}"
assert all(output.shape[-1] == 8 for output in outputs), \
    "Expected 5 box/objectness values + 3 class scores."

print("\nFULL MODEL SHAPE CHECK: PASS")


/kaggle/working/yolov5_repo
Created: /kaggle/working/yolov5_repo/models/stormsight_cctsdb.yaml

                 from  n    params  module                                  arguments                     
  0                -1  1      3520  models.common.Conv                      [3, 32, 6, 2, 2]              
  1                -1  1     18560  models.common.Conv                      [32, 64, 3, 2]                
  2                -1  1     18816  models.common.C3                        [64, 64, 1]                   
  3                -1  1     73984  models.common.Conv                      [64, 128, 3, 2]               
  4                -1  2    115712  models.common.C3                        [128, 128, 2]                 
  5                -1  1    295424  models.common.Conv                      [128, 256, 3, 2]              
  6                -1  3    625152  models.common.C3                        [256, 256, 3]                 
  7                -1  1   1180672  models.commo

In [44]:

%cd /kaggle/working/yolov5_repo

from pathlib import Path

data_yaml = Path("/kaggle/working/stormsight_data/stormsight_data.yaml")
model_yaml = Path("models/stormsight_cctsdb.yaml")

assert data_yaml.exists(), f"Dataset YAML missing: {data_yaml}"
assert model_yaml.exists(), f"Model YAML missing: {model_yaml}"

print("Model YAML:", model_yaml)
print("Dataset YAML:", data_yaml)
print("Starting one-epoch smoke test...")

!python train.py \
  --cfg models/stormsight_cctsdb.yaml \
  --data /kaggle/working/stormsight_data/stormsight_data.yaml \
  --weights '' \
  --hyp data/hyps/hyp.scratch-low.yaml \
  --epochs 1 \
  --batch-size 4 \
  --img 320 \
  --workers 2 \
  --device 0 \
  --project /kaggle/working/stormsight_checkpoints \
  --name stormsight_smoke \
  --save-period 1 \
  --exist-ok


/kaggle/working/yolov5_repo
Model YAML: models/stormsight_cctsdb.yaml
Dataset YAML: /kaggle/working/stormsight_data/stormsight_data.yaml
Starting one-epoch smoke test...
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice: (30 second timeout) 
wandb: WARNING W&B disabled due to login timeout.
wandb: ERROR Error while calling W&B API: auth token too short (<Response [401]>)
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
train: weights=, cfg=models/stormsight_cctsdb.yaml, data=/kaggle/working/stormsight_data/stormsight_data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=1, batch_size=4, imgsz=320, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=d

In [45]:

from pathlib import Path

run_dir = Path(
    "/kaggle/working/stormsight_checkpoints/stormsight_smoke"
)
weights_dir = run_dir / "weights"

for name in ["last.pt", "best.pt"]:
    path = weights_dir / name
    print(f"{name}: {'FOUND' if path.exists() else 'NOT FOUND'}")


last.pt: FOUND
best.pt: FOUND


In [46]:

%cd /kaggle/working/yolov5_repo

!WANDB_MODE=disabled python train.py \
  --cfg models/stormsight_cctsdb.yaml \
  --data /kaggle/working/stormsight_data/stormsight_data.yaml \
  --weights yolov5s.pt \
  --hyp data/hyps/hyp.scratch-low.yaml \
  --epochs 50 \
  --batch-size 8 \
  --img 416 \
  --workers 2 \
  --device 0 \
  --project /kaggle/working/stormsight_checkpoints \
  --name stormsight_pretrained_40 \
  --save-period 1 \
  --exist-ok


/kaggle/working/yolov5_repo
WARNING ⚠️ wandb: wandb is deprecated and will be removed in a future release. See supported integrations at https://github.com/ultralytics/yolov5#integrations.
train: weights=yolov5s.pt, cfg=models/stormsight_cctsdb.yaml, data=/kaggle/working/stormsight_data/stormsight_data.yaml, hyp=data/hyps/hyp.scratch-low.yaml, epochs=50, batch_size=8, imgsz=416, rect=False, resume=False, nosave=False, noval=False, noautoanchor=False, noplots=False, evolve=None, evolve_population=data/hyps, resume_evolve=None, bucket=, cache=None, image_weights=False, device=0, multi_scale=False, single_cls=False, optimizer=SGD, sync_bn=False, workers=2, project=/kaggle/working/stormsight_checkpoints, name=stormsight_pretrained_40, exist_ok=True, quad=False, cos_lr=False, label_smoothing=0.0, patience=100, freeze=[0], save_period=1, seed=0, local_rank=-1, entity=None, upload_dataset=False, bbox_interval=-1
github: up to date with https://github.com/ultralytics/yolov5 ✅
YOLOv5 🚀 v7.0-561

In [47]:

from pathlib import Path

weights_dir = Path(
    "/kaggle/working/stormsight_checkpoints/"
    "stormsight_pretrained_40/weights"
)

for name in ["last.pt", "best.pt"]:
    p = weights_dir / name
    print(f"{name}: {'FOUND' if p.exists() else 'NOT FOUND'}")
    if p.exists():
        print(f"  Size: {p.stat().st_size / (1024 * 1024):.1f} MB")


last.pt: FOUND
  Size: 19.2 MB
best.pt: FOUND
  Size: 19.2 MB


In [51]:
# StormSight: F1-score and estimated FPS from test results

precision = 0.924
recall = 0.868

# F1-score
f1 = 2 * precision * recall / (precision + recall)

# Timing reported by your Kaggle val.py output (milliseconds/image)
preprocess_ms = 0.1
inference_ms = 3.4
nms_ms = 1.1

fps_inference = 1000 / inference_ms
fps_end_to_end = 1000 / (
    preprocess_ms + inference_ms + nms_ms
)

print("=" * 45)
print("       StormSight TEST METRICS")
print("=" * 45)
print(f"Precision       : {precision:.4f} ({precision*100:.2f}%)")
print(f"Recall          : {recall:.4f} ({recall*100:.2f}%)")
print(f"F1-score        : {f1:.4f} ({f1*100:.2f}%)")
print(f"Inference FPS   : {fps_inference:.2f}")
print(f"Estimated total FPS: {fps_end_to_end:.2f}")
print("=" * 45)

       StormSight TEST METRICS
Precision       : 0.9240 (92.40%)
Recall          : 0.8680 (86.80%)
F1-score        : 0.8951 (89.51%)
Inference FPS   : 294.12
Estimated total FPS: 217.39


In [67]:

import os

weights_path = (
    "/kaggle/working/stormsight_checkpoints/"
    "stormsight_pretrained_40/weights/best.pt"
)

print("Checkpoint exists:", os.path.exists(weights_path))

if os.path.exists(weights_path):
    print("Checkpoint size:", round(os.path.getsize(weights_path) / (1024**2), 2), "MB")
    print("Checkpoint saved at:", weights_path)
else:
    print("best.pt not found. Check your training output directory.")


Checkpoint exists: True
Checkpoint size: 19.24 MB
Checkpoint saved at: /kaggle/working/stormsight_checkpoints/stormsight_pretrained_40/weights/best.pt


In [68]:
from pathlib import Path

locations = [
    Path("/kaggle/working/stormsight_checkpoints/stormsight_pretrained_40/weights/best.pt"),
    Path("/kaggle/working/best.pt"),
]

for p in locations:
    print(f"{p}: {'FOUND' if p.is_file() else 'NOT FOUND'}", flush=True)

/kaggle/working/stormsight_checkpoints/stormsight_pretrained_40/weights/best.pt: FOUND
/kaggle/working/best.pt: NOT FOUND


In [70]:

from pathlib import Path
import shutil

source = Path(
    "/kaggle/working/stormsight_checkpoints/"
    "stormsight_pretrained_40/weights/best.pt"
)
destination = Path("/kaggle/working/best.pt")

if source.is_file():
    shutil.copy2(source, destination)
    print("Checkpoint copied successfully!")
    print("Saved at:", destination)
    print("Size:", round(destination.stat().st_size / (1024 * 1024), 2), "MB")
else:
    print("Source checkpoint not found.")


Checkpoint copied successfully!
Saved at: /kaggle/working/best.pt
Size: 19.24 MB


In [72]:

from pathlib import Path

repo = Path("/kaggle/working/yolov5_repo")

required = [
    "detect.py",
    "models/stormsight_cctsdb.yaml",
    "models/common.py",
    "models/yolo.py",
    "utils/loss.py",
]

for item in required:
    p = repo / item
    print(f"{'FOUND' if p.is_file() else 'MISSING'}: {item}")


FOUND: detect.py
FOUND: models/stormsight_cctsdb.yaml
FOUND: models/common.py
FOUND: models/yolo.py
FOUND: utils/loss.py


In [2]:
%%writefile /kaggle/working/yolov5_repo/app.py

from pathlib import Path
import subprocess
import sys
import time

import gradio as gr

REPO = Path("/kaggle/working/yolov5_repo")
WEIGHTS = Path("/kaggle/working/best.pt")
OUTPUT = Path("/kaggle/working/stormsight_web")

OUTPUT.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["mandatory", "prohibitory", "warning"]


def detect_signs(image_path):
    if not image_path:
        raise gr.Error("Please upload a road image.")

    if not WEIGHTS.is_file():
        raise gr.Error(f"Model checkpoint not found: {WEIGHTS}")

    image_path = Path(image_path)
    run_name = f"prediction_{time.time_ns()}"
    run_dir = OUTPUT / run_name

    command = [
        sys.executable,
        str(REPO / "detect.py"),
        "--weights", str(WEIGHTS),
        "--source", str(image_path),
        "--imgsz", "416",
        "--conf-thres", "0.25",
        "--device", "0",
        "--project", str(OUTPUT),
        "--name", run_name,
        "--save-txt",
        "--save-conf",
    ]

    start = time.perf_counter()

    result = subprocess.run(
        command,
        cwd=str(REPO),
        capture_output=True,
        text=True,
        timeout=180,
    )

    elapsed = time.perf_counter() - start

    if result.returncode != 0:
        error = (result.stderr or result.stdout or "Unknown error")[-3000:]
        raise gr.Error("Model detection failed:\n" + error)

    annotated = run_dir / image_path.name

    if not annotated.is_file():
        candidates = [
            p for p in run_dir.rglob("*")
            if p.is_file()
            and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        ]
        if not candidates:
            raise gr.Error(
                "Detection finished, but the annotated image was not found.\n"
                + result.stdout[-1500:]
            )
        annotated = candidates[0]

    label_file = run_dir / "labels" / f"{image_path.stem}.txt"

    if label_file.is_file():
        lines = [
            line.strip()
            for line in label_file.read_text().splitlines()
            if line.strip()
        ]

        details = []
        for line in lines:
            values = line.split()
            try:
                class_id = int(values[0])
                class_name = (
                    CLASS_NAMES[class_id]
                    if 0 <= class_id < len(CLASS_NAMES)
                    else f"class_{class_id}"
                )
                confidence = float(values[-1]) if len(values) >= 6 else None

                if confidence is not None:
                    details.append(
                        f"- {class_name}: confidence {confidence:.3f}"
                    )
                else:
                    details.append(f"- {class_name}: confidence unavailable")
            except (ValueError, IndexError):
                details.append(f"- Detection: {line}")

        summary = (
            f"Detections: {len(lines)}\n"
            + ("\n".join(details) if details else "No signs detected.")
        )
    else:
        summary = "No traffic signs detected above the confidence threshold."

    summary += (
        f"\n\nProcessing time: {elapsed:.2f} seconds "
        "(includes detector startup and image processing)."
    )

    return str(annotated), summary


with gr.Blocks(title="StormSight Traffic Sign Detection") as demo:
    gr.Markdown(
        "# StormSight — Traffic Sign Detection\n"
        "Upload a road image to detect traffic signs using your trained model.\n\n"
        "**Classes:** mandatory · prohibitory · warning"
    )

    with gr.Row():
        input_image = gr.Image(
            type="filepath",
            label="Upload road image"
        )
        output_image = gr.Image(
            type="filepath",
            label="Detected traffic signs"
        )

    detect_button = gr.Button(
        "Detect traffic signs",
        variant="primary"
    )
    details = gr.Textbox(
        label="Prediction details",
        lines=8
    )

    detect_button.click(
        fn=detect_signs,
        inputs=input_image,
        outputs=[output_image, details]
    )

if __name__ == "__main__":
    demo.launch(
        server_name="0.0.0.0",
        server_port=7860,
        share=True
    )

Writing /kaggle/working/yolov5_repo/app.py


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/yolov5_repo/app.py'

In [3]:

from pathlib import Path
import zipfile

REPO = Path("/kaggle/working/yolov5_repo")
WEIGHTS = Path("/kaggle/working/best.pt")
ZIP_PATH = Path("/kaggle/working/StormSight_deployment.zip")

assert REPO.is_dir(), f"Repository not found: {REPO}"
assert WEIGHTS.is_file(), f"Trained model not found: {WEIGHTS}"

skip_dirs = {
    ".git", "__pycache__", "runs", "datasets",
    "wandb", ".pytest_cache", ".mypy_cache"
}

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for file in REPO.rglob("*"):
        if not file.is_file():
            continue
        relative = file.relative_to(REPO)
        if any(part in skip_dirs for part in relative.parts):
            continue
        if file.suffix in {".pyc", ".pyo"}:
            continue
        z.write(file, arcname=f"StormSight/{relative.as_posix()}")

    z.write(WEIGHTS, arcname="StormSight/best.pt")

print("Deployment backup created:", ZIP_PATH)
print(f"ZIP size: {ZIP_PATH.stat().st_size / (1024*1024):.2f} MB")
print("Includes the YOLOv5 repository, custom code, app files and best.pt.")


AssertionError: Repository not found: /kaggle/working/yolov5_repo

In [6]:

from pathlib import Path

paths = {
    "Repository": Path("/kaggle/working/yolov5_repo"),
    "Trained checkpoint": Path("/kaggle/working/best.pt"),
    "Custom model YAML": Path("/kaggle/working/yolov5_repo/models/stormsight_cctsdb.yaml"),
    "Gradio app": Path("/kaggle/working/yolov5_repo/app.py"),
}

for name, path in paths.items():
    print(f"{name}: {'FOUND' if path.exists() else 'MISSING'}")
    if path.is_file():
        print(f"  Size: {path.stat().st_size / (1024 * 1024):.2f} MB")

repo = paths["Repository"]
if repo.is_dir():
    print("\nCustom files:")
    for rel in [
        "models/common.py",
        "models/yolo.py",
        "utils/loss.py",
        "app.py",
    ]:
        p = repo / rel
        print(f"{rel}: {'FOUND' if p.is_file() else 'MISSING'}")


Repository: MISSING
Trained checkpoint: MISSING
Custom model YAML: MISSING
Gradio app: MISSING


In [4]:

from pathlib import Path

roots = [
    Path("/kaggle/working"),
    Path("/kaggle/input"),
]

for root in roots:
    print(f"\n--- {root} ---")
    if not root.exists():
        print("Directory does not exist")
        continue

    for item in list(root.iterdir())[:40]:
        print(("[DIR] " if item.is_dir() else "[FILE] ") + item.name)

print("\n--- Searching for model checkpoints ---")
for root in roots:
    if root.exists():
        for p in root.rglob("best.pt"):
            print(p)

print("\n--- Searching for YOLOv5 repositories ---")
for root in roots:
    if root.exists():
        for p in root.rglob("detect.py"):
            if p.parent.name.lower() in {"yolov5_repo", "yolov5"}:
                print(p.parent)



--- /kaggle/working ---
[DIR] .virtual_documents

--- /kaggle/input ---
[DIR] datasets

--- Searching for model checkpoints ---

--- Searching for YOLOv5 repositories ---


In [83]:
from pathlib import Path

app_file = Path("/kaggle/working/yolov5_repo/app.py")
code = app_file.read_text()

old = """        share=True
    )"""

new = """        share=True,
        allowed_paths=[str(OUTPUT)]
    )"""

if old in code:
    code = code.replace(old, new, 1)
    app_file.write_text(code)
    print("Fixed: Gradio can now access prediction output files.")
elif "allowed_paths=[str(OUTPUT)]" in code:
    print("The allowed_paths setting is already present.")
else:
    print("Automatic edit could not find the launch settings.")
    print("Open app.py and add allowed_paths to demo.launch().")

The allowed_paths setting is already present.


In [84]:

%cd /kaggle/working/yolov5_repo
!python app.py


/kaggle/working/yolov5_repo
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://f30c8a4a6e77c68746.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
^C
Keyboard interruption in main thread... closing server.
Killing tunnel 0.0.0.0:7860 <> https://f30c8a4a6e77c68746.gradio.live
